In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/mnt/c/Users/suraj/MyWorkplace/AITraining/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

student_model_path = "./models/functiongemma-270m-it-kd-2"
student_tokenizer = AutoTokenizer.from_pretrained(student_model_path)
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_path,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
student_model.to(device)
print("Student model device:", next(student_model.parameters()).device)

cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1868.06it/s]


Student model device: cuda:0


In [3]:
from datasets import load_dataset
data_file = "../AI/dataset/tool_dataset_all.jsonl"

raw_dataset = load_dataset("json", data_files=data_file, split="train")
tools = raw_dataset[0]["tools"]
developer = raw_dataset[0]["messages"][0]['content']
print(raw_dataset[0])
print(tools)
print(developer)

{'metadata': 'test', 'tools': [{'type': 'function', 'function': {'name': 'toggle_lights', 'description': 'Turn lights on or off in a specified room.', 'parameters': {'type': 'object', 'properties': {'room': {'type': 'string', 'enum': ['living_room', 'bedroom', 'kitchen', 'bathroom', 'office', 'hallway'], 'description': 'The room whose lights to control.'}, 'state': {'type': 'string', 'enum': ['on', 'off'], 'description': 'Whether to turn lights on or off.'}}, 'required': []}}}, {'type': 'function', 'function': {'name': 'set_thermostat', 'description': 'Set the temperature for heating or cooling.', 'parameters': {'type': 'object', 'properties': {'temperature': {'type': 'integer', 'minimum': 60, 'maximum': 80, 'description': 'The target temperature in degrees Fahrenheit (60-80).'}, 'mode': {'type': 'string', 'enum': ['heat', 'cool', 'auto'], 'description': 'The thermostat mode: heat, cool, or auto.'}}, 'required': []}}}, {'type': 'function', 'function': {'name': 'lock_door', 'description

In [4]:
print("Chat interface ready! Type 'exit' to quit.\n")

while True:
    user_input = input("user: ")
    if user_input.lower() == "exit":
        break

    messages = [
        {"role": "developer", "content": developer},
        {"role": "user", "content": user_input},
    ]

    prompt = student_tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True
    )
    # Encode user input
    inputs = student_tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate model output
    with torch.no_grad():
        outputs = student_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=1,
            top_p=0.9
        )

    # Decode raw output string
    response = student_tokenizer.decode(outputs[0], skip_special_tokens=False)
    print("model:", response)

Chat interface ready! Type 'exit' to quit.



[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


model: <bos><bos><start_of_turn>developer
You are an on-device smart home controller. Given a natural language command from the user, call the appropriate smart home function. If the user does not specify a required value (e.g. which room or what temperature), omit that parameter from the function call. Maintain context across conversation turns to resolve pronouns and sequential commands.<start_function_declaration>declaration:toggle_lights{description:<escape>Turn lights on or off in a specified room.<escape>,parameters:{properties:{room:{description:<escape>The room whose lights to control.<escape>,enum:[<escape>living_room<escape>,<escape>bedroom<escape>,<escape>kitchen<escape>,<escape>bathroom<escape>,<escape>office<escape>,<escape>hallway<escape>],type:<escape>STRING<escape>},state:{description:<escape>Whether to turn lights on or off.<escape>,enum:[<escape>on<escape>,<escape>off<escape>],type:<escape>STRING<escape>}},type:<escape>OBJECT<escape>}}<end_function_declaration><start_

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


model: <bos><bos><start_of_turn>developer
You are an on-device smart home controller. Given a natural language command from the user, call the appropriate smart home function. If the user does not specify a required value (e.g. which room or what temperature), omit that parameter from the function call. Maintain context across conversation turns to resolve pronouns and sequential commands.<start_function_declaration>declaration:toggle_lights{description:<escape>Turn lights on or off in a specified room.<escape>,parameters:{properties:{room:{description:<escape>The room whose lights to control.<escape>,enum:[<escape>living_room<escape>,<escape>bedroom<escape>,<escape>kitchen<escape>,<escape>bathroom<escape>,<escape>office<escape>,<escape>hallway<escape>],type:<escape>STRING<escape>},state:{description:<escape>Whether to turn lights on or off.<escape>,enum:[<escape>on<escape>,<escape>off<escape>],type:<escape>STRING<escape>}},type:<escape>OBJECT<escape>}}<end_function_declaration><start_

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


model: <bos><bos><start_of_turn>developer
You are an on-device smart home controller. Given a natural language command from the user, call the appropriate smart home function. If the user does not specify a required value (e.g. which room or what temperature), omit that parameter from the function call. Maintain context across conversation turns to resolve pronouns and sequential commands.<start_function_declaration>declaration:toggle_lights{description:<escape>Turn lights on or off in a specified room.<escape>,parameters:{properties:{room:{description:<escape>The room whose lights to control.<escape>,enum:[<escape>living_room<escape>,<escape>bedroom<escape>,<escape>kitchen<escape>,<escape>bathroom<escape>,<escape>office<escape>,<escape>hallway<escape>],type:<escape>STRING<escape>},state:{description:<escape>Whether to turn lights on or off.<escape>,enum:[<escape>on<escape>,<escape>off<escape>],type:<escape>STRING<escape>}},type:<escape>OBJECT<escape>}}<end_function_declaration><start_

In [5]:
prompt = "Hello"
inputs = student_tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = student_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
    )

print(student_tokenizer.decode(outputs[0], skip_special_tokens=True))


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Hello!

I am unable to assist with this request at this moment. My current capabilities are limited to managing virtual assistant tasks. I cannot access or process real-time information or generate content related to gaming or esports.
